# Mean-Variance Portfolios

Markowitz (1952) chose portfolios by expected return and variance alone. With mean returns $\mu$ and covariance $\Sigma$, weights $w$ give expected return $w'\mu$ and variance $w'\Sigma w$. This notebook finds the minimum-variance and the maximum Sharpe portfolios from sample estimates, draws the efficient frontier, and compares in-sample with true expected returns.

In [ ]:
Pollis.packages("PortfolioOptimisers", "Clarabel", "Plots")

In [ ]:
using PortfolioOptimisers, Clarabel, Random, Statistics, Plots

## The data

1,000 trading days of daily simple returns for 8 assets in two sectors. A market factor moves every asset, a sector factor moves each sector, and Student-t shocks with 5 degrees of freedom give fat tails. The true annualised means are 12% for Tech and 6% for Util, so the estimates can be checked against them.

In [ ]:
Random.seed!(42)
T, N = 1_000, 8
t5(n) = sqrt(3 / 5) .* randn(n) ./ sqrt.(vec(sum(abs2, randn(5, n); dims = 1)) ./ 5)   # unit-variance Student-t(5) shocks
mkt = 0.008 .* t5(T); tech = 0.010 .* t5(T); util = 0.005 .* t5(T)   # market and sector factors
mu_d = [fill(0.12, 4); fill(0.06, 4)] ./ 252   # true daily means
R = hcat([j <= 4 ? mu_d[j] .+ 1.2 .* mkt .+ tech .+ 0.012 .* t5(T) : mu_d[j] .+ 0.6 .* mkt .+ util .+ 0.006 .* t5(T) for j in 1:N]...)
asset_names = ["Tech1", "Tech2", "Tech3", "Tech4", "Util1", "Util2", "Util3", "Util4"]

rd = ReturnsResult(; nx = asset_names, X = R)
rf_daily = 0.02 / 252
slv = Solver(; name = :clarabel, solver = Clarabel.Optimizer, settings = Dict("verbose" => false),
             check_sol = (; allow_local = true, allow_almost = true))
show_weights(w) = foreach((nm, wi) -> println(rpad(nm, 8), lpad(round(100 * wi; digits = 2), 8), " %"), asset_names, w)

## Minimum variance

The minimum-variance portfolio uses the covariance only. Weights are long-only and add up to 1.

In [ ]:
w_minv = optimise(MeanRisk(; opt = JuMPOptimiser(; slv = slv), r = Variance(), obj = MinimumRisk()), rd).w
show_weights(w_minv)

## Maximum Sharpe ratio

The tangency portfolio maximises $(w'\mu - r_f) / \sqrt{w'\Sigma w}$, so it uses the sample means as well.

In [ ]:
res = optimise(MeanRisk(; opt = JuMPOptimiser(; slv = slv), r = StandardDeviation(), obj = MaximumRatio(; rf = rf_daily)), rd)
w_tan, pr = res.w, res.pr
show_weights(w_tan)

## The efficient frontier

Thirty portfolios of minimum variance at rising return floors trace the frontier of the sample.

In [ ]:
ret_f = ArithmeticReturn(; settings = JuMPReturnsSettings(; lb = Frontier(; N = 30)))
fr = optimise(MeanRisk(; opt = JuMPOptimiser(; slv = slv, ret = ret_f), r = Variance()), rd)
ok = [rc isa OptimisationSuccess for rc in fr.retcode]
ann_vol(v) = sqrt(252) * expected_risk(StandardDeviation(), v, pr)
ann_ret(v) = 252 * expected_return(ArithmeticReturn(), v, pr)
plot(ann_vol.(fr.w[ok]), ann_ret.(fr.w[ok]); label = "Efficient Frontier", lw = 2,
     xlabel = "Annualised Volatility", ylabel = "Annualised Mean Return")
scatter!([ann_vol(w_minv)], [ann_ret(w_minv)]; label = "Minimum Variance")
scatter!([ann_vol(w_tan)], [ann_ret(w_tan)]; label = "Maximum Sharpe")

## In-sample against true returns

The data was simulated, so the true expected return of any weights is known. The gap between the two columns is estimation error.

In [ ]:
for (nm, w) in [("Minimum Variance", w_minv), ("Maximum Sharpe", w_tan), ("Equal Weight", fill(1 / N, N))]
    println(rpad(nm, 18), "in-sample ", lpad(round(100 * ann_ret(w); digits = 2), 6), " %   true ", lpad(round(100 * 252 * sum(w .* mu_d); digits = 2), 6), " %")
end

## Exercises

1. Change the seed to 1 and run again. Which weights change more, those of the minimum-variance or of the maximum Sharpe portfolio?
2. Replace `MaximumRatio(; rf = rf_daily)` with `MaximumUtility(; l = 2)` and `r = Variance()`. Try `l = 10` and `l = 0.5`: where on the frontier does each portfolio lie?
3. Set `T = 250` and compare the gap between in-sample and true returns with that for `T = 1_000`.